# Extração SIGTAP — Tabela Unificada de Procedimentos · Conecta Saúde

O SIGTAP é a **tabela de domínio dos procedimentos do SUS**. Ele não traz volume nem produção: traz o significado dos códigos que o SIH/SUS.

Sem ele, o `PROC_REA` do SIH é o número `0303140151` e nada mais. Com ele, é "tratamento de insuficiência cardíaca", pertencente a procedimentos clínicos, de média complexidade, financiado pelo MAC. É a diferença entre um painel que lista códigos e um painel que um gestor consegue ler.


## 1. Papel do SIGTAP no projeto

O SIGTAP é uma **dimensão**, não um fato. Ele entra em todo cruzamento onde exista código de procedimento:

| Cruzamento | Chave | O que passa a ser possível |
|---|---|---|
| SIGTAP × SIH/SUS | `PROC_REA` | Nomear e agrupar os procedimentos que mais internam |


Também é o que permite separar **alta e média complexidade**, recorte central para medir acesso: um município que só realiza atenção básica depende inteiramente da rede regional para o resto.

## 2. Endpoint público

```text
ftp://ftp2.datasus.gov.br/pub/sistemas/tup/downloads/TabelaUnificada_{AAAAMM}_v{versao}.zip
```

In [ ]:
# A ftplib e a zipfile já vêm com o Python. Só pandas e pyarrow precisam de instalação.
%pip install -q pandas pyarrow

## 3. Parâmetros da extração

In [1]:
import zipfile
from ftplib import FTP
from pathlib import Path

import pandas as pd

# Competência de referência. Use a mesma do CNES e do SIH para que os
# códigos de procedimento correspondam ao período analisado.
COMPETENCIA = "202412"

FTP_HOST_SIGTAP = "ftp2.datasus.gov.br"
DIR_SIGTAP = "/pub/sistemas/tup/downloads"


def raiz_do_projeto() -> Path:
    '''Devolve a pasta do repositório, subindo até encontrar o .git.'''
    atual = Path.cwd().resolve()
    for pasta in (atual, *atual.parents):
        if (pasta / ".git").exists():
            return pasta
    return atual


RAIZ = raiz_do_projeto()
DIRETORIO_RAW = RAIZ / "dados" / "raw" / "sigtap"
DIRETORIO_TRATADO = RAIZ / "dados" / "tratado" / "sigtap"

for pasta in (DIRETORIO_RAW, DIRETORIO_TRATADO):
    pasta.mkdir(parents=True, exist_ok=True)

print(f"Raiz do projeto : {RAIZ}")
print(f"Competência     : {COMPETENCIA}")
print(f"Saída tratada   : {DIRETORIO_TRATADO}")

Raiz do projeto : C:\Users\Vitor Nobre\Documents\Workspace\conecta-saude
Competência     : 202412
Saída tratada   : C:\Users\Vitor Nobre\Documents\Workspace\conecta-saude\dados\tratado\sigtap


## 4. Localização e download do pacote

Como o sufixo de versão é imprevisível, a função lista o diretório e filtra pelo prefixo da competência. Se houver mais de uma versão publicada para o mesmo mês, acontece quando o Ministério republica a tabela, fica a mais recente pelo nome, que é o que o próprio SIGTAP considera vigente.

In [2]:
def localizar_pacote(competencia: str) -> str:
    '''Descobre o nome do ZIP da competência, já que a versão não é previsível.'''
    ftp = FTP(FTP_HOST_SIGTAP, timeout=300)
    ftp.login()
    ftp.set_pasv(True)
    try:
        ftp.cwd(DIR_SIGTAP)
        prefixo = f"TabelaUnificada_{competencia}_"
        candidatos = sorted(n for n in ftp.nlst() if n.startswith(prefixo))
    finally:
        ftp.quit()

    if not candidatos:
        raise FileNotFoundError(
            f"Nenhum pacote para a competência {competencia} em {DIR_SIGTAP}"
        )
    if len(candidatos) > 1:
        print(f"  {len(candidatos)} versões publicadas; usando a mais recente")
    return candidatos[-1]


def baixar_pacote(competencia: str) -> Path:
    '''Baixa o ZIP da competência, reaproveitando o que já estiver em disco.'''
    nome = localizar_pacote(competencia)
    destino = DIRETORIO_RAW / nome
    if destino.exists() and destino.stat().st_size > 0:
        print(f"Já em cache: {nome}")
        return destino

    ftp = FTP(FTP_HOST_SIGTAP, timeout=300)
    ftp.login()
    ftp.set_pasv(True)
    try:
        ftp.cwd(DIR_SIGTAP)
        with open(destino, "wb") as arquivo:
            ftp.retrbinary(f"RETR {nome}", arquivo.write, blocksize=65536)
    finally:
        ftp.quit()

    print(f"Baixado: {nome} ({destino.stat().st_size / 1024:.0f} KB)")
    return destino


caminho_zip = baixar_pacote(COMPETENCIA)

with zipfile.ZipFile(caminho_zip) as pacote:
    membros = pacote.infolist()

print(f"\n{len(membros)} arquivos no pacote. Os 12 maiores:")
for item in sorted(membros, key=lambda x: -x.file_size)[:12]:
    print(f"  {item.filename:<40} {item.file_size:>12,} bytes")

Já em cache: TabelaUnificada_202412_v2501172121.zip

87 arquivos no pacote. Os 12 maiores:
  tb_descricao.txt                           16,204,594 bytes
  rl_procedimento_ocupacao.txt                4,567,416 bytes
  tb_tuss.txt                                 2,663,892 bytes
  rl_procedimento_cid.txt                     1,864,932 bytes
  tb_cid.txt                                  1,609,346 bytes
  tb_procedimento.txt                         1,608,208 bytes
  tb_sia_sih.txt                                997,577 bytes
  rl_procedimento_compativel.txt                439,671 bytes
  tb_ocupacao.txt                               429,444 bytes
  rl_procedimento_detalhe.txt                   190,533 bytes
  tb_descricao_detalhe.txt                      168,462 bytes
  rl_procedimento_habilitacao.txt               166,530 bytes


## 5. Leitor genérico guiado pelo layout

Cada tabela vem em dois arquivos:

```text
tb_procedimento.txt          os dados, largura fixa, sem cabeçalho
tb_procedimento_layout.txt   a descrição das colunas
```

E o layout é legível por máquina:

```text
Coluna,Tamanho,Inicio,Fim,Tipo
CO_PROCEDIMENTO,10,1,10,VARCHAR2
NO_PROCEDIMENTO,250,11,260,VARCHAR2
TP_COMPLEXIDADE,1,261,261,VARCHAR2
```

In [3]:
def ler_layout(pacote: zipfile.ZipFile, tabela: str) -> pd.DataFrame:
    '''Lê o arquivo _layout.txt que descreve as colunas de uma tabela.'''
    texto = pacote.read(f"{tabela}_layout.txt").decode("latin-1")
    linhas = [l for l in texto.splitlines() if l.strip()]

    campos = []
    for linha in linhas[1:]:            # a primeira linha é o cabeçalho
        coluna, tamanho, inicio, fim, tipo = linha.split(",")[:5]
        campos.append({
            "coluna": coluna.strip(),
            "inicio": int(inicio),
            "fim": int(fim),
            "tipo": tipo.strip(),
        })
    return pd.DataFrame(campos)


def ler_tabela(pacote: zipfile.ZipFile, tabela: str) -> pd.DataFrame:
    '''Lê uma tabela de largura fixa do SIGTAP usando o layout declarado.'''
    layout = ler_layout(pacote, tabela)
    linhas = pacote.read(f"{tabela}.txt").decode("latin-1").splitlines()

    dados = {}
    for campo in layout.itertuples():
        # inicio/fim vêm em base 1 e são inclusivos; Python fatia em base 0.
        recorte = [linha[campo.inicio - 1:campo.fim].strip() for linha in linhas]
        if campo.tipo == "NUMBER":
            dados[campo.coluna] = pd.to_numeric(recorte, errors="coerce")
        else:
            dados[campo.coluna] = recorte

    return pd.DataFrame(dados)


with zipfile.ZipFile(caminho_zip) as pacote:
    print("Layout de tb_procedimento:")
    display(ler_layout(pacote, "tb_procedimento"))

Layout de tb_procedimento:


,coluna,inicio,fim,tipo
0,CO_PROCEDIMENTO,1,10,VARCHAR2
1,NO_PROCEDIMENTO,11,260,VARCHAR2
2,TP_COMPLEXIDADE,261,261,VARCHAR2
3,TP_SEXO,262,262,VARCHAR2
4,QT_MAXIMA_EXECUCAO,263,266,NUMBER
5,QT_DIAS_PERMANENCIA,267,270,NUMBER
6,QT_PONTOS,271,274,NUMBER
7,VL_IDADE_MINIMA,275,278,NUMBER
8,VL_IDADE_MAXIMA,279,282,NUMBER
9,VL_SH,283,292,NUMBER


## 6. Leitura das tabelas do MVP

Seis tabelas bastam para o projeto. As outras 36 do pacote como CID, ocupações, habilitações e compatibilidades, ficam fora do escopo.

| Tabela | Conteúdo | Linhas |
|---|---|---|
| `tb_procedimento` | Procedimentos, valores e complexidade | 4.844 |
| `tb_grupo` | Grupo (1º nível) | 9 |
| `tb_sub_grupo` | Subgrupo (2º nível) | 67 |
| `tb_forma_organizacao` | Forma de organização (3º nível) | 409 |
| `tb_financiamento` | Fonte de financiamento | 7 |
| `tb_modalidade` | Ambulatorial, hospitalar, hospital-dia, domiciliar | 4 |

In [4]:
TABELAS = [
    "tb_procedimento", "tb_grupo", "tb_sub_grupo",
    "tb_forma_organizacao", "tb_financiamento", "tb_modalidade",
]

with zipfile.ZipFile(caminho_zip) as pacote:
    sigtap = {t: ler_tabela(pacote, t) for t in TABELAS}

for nome, df in sigtap.items():
    print(f"  {nome:<24} {len(df):>6,} linhas x {df.shape[1]} colunas")

sigtap["tb_procedimento"].head()

  tb_procedimento           4,844 linhas x 16 colunas
  tb_grupo                      9 linhas x 3 colunas
  tb_sub_grupo                 67 linhas x 4 colunas
  tb_forma_organizacao        409 linhas x 5 colunas
  tb_financiamento              7 linhas x 3 colunas
  tb_modalidade                 4 linhas x 3 colunas


,CO_PROCEDIMENTO,NO_PROCEDIMENTO,TP_COMPLEXIDADE,TP_SEXO,QT_MAXIMA_EXECUCAO,QT_DIAS_PERMANENCIA,QT_PONTOS,VL_IDADE_MINIMA,VL_IDADE_MAXIMA,VL_SH,VL_SA,VL_SP,CO_FINANCIAMENTO,CO_RUBRICA,QT_TEMPO_PERMANENCIA,DT_COMPETENCIA
0,0101010010,ATIVIDADE EDUCATIVA / ORIENTAÇÃO EM GRUPO NA A...,1,N,9999,9999,0,9999,9999,0,0,0,01,,9999,202412
1,0101010028,ATIVIDADE EDUCATIVA / ORIENTAÇÃO EM GRUPO NA A...,2,I,9999,9999,0,84,1571,0,270,0,06,,9999,202412
2,0101010036,PRÁTICA CORPORAL / ATIVIDADE FÍSICA EM GRUPO,1,I,9999,9999,0,72,1571,0,0,0,01,,9999,202412
3,0101010095,PREVENÇÃO DA COVID-19 NAS ESCOLAS,1,I,9999,9999,0,0,1571,0,0,0,01,,9999,202412
4,0101010109,ATIVIDADES EDUCATIVAS DA POPULAÇÃO SOBRE A TEM...,1,I,9999,9999,0,0,1571,0,0,0,07,,9999,202412


## 7. A hierarquia está dentro do próprio código

O código de procedimento não é um identificador opaco: ele **é** a hierarquia, concatenada.

```text
0 3 . 0 3 . 1 4 . 0 1 5 - 1
└─┬─┘ └─┬─┘ └─┬─┘ └──┬──┘ ┬
grupo  sub   forma  seq  dígito
 03     03    14    015    1
```

`0303140151` = grupo 03 (procedimentos clínicos), subgrupo 03, forma de organização 14, sequencial 015.

Isso significa que o enriquecimento não precisa de tabela de-para: bastam três fatias do código. E significa também que **um filtro por prefixo é um filtro por categoria**, `PROC_REA` começando com `04` são todas as cirurgias, sem precisar de join nenhum.

In [6]:
procedimentos = sigtap["tb_procedimento"].copy()

# As três fatias que reconstroem a hierarquia a partir do código.
procedimentos["CO_GRUPO"] = procedimentos["CO_PROCEDIMENTO"].str[0:2]
procedimentos["CO_SUB_GRUPO"] = procedimentos["CO_PROCEDIMENTO"].str[2:4]
procedimentos["CO_FORMA_ORGANIZACAO"] = procedimentos["CO_PROCEDIMENTO"].str[4:6]

antes = len(procedimentos)
procedimentos = (
    procedimentos
    .merge(sigtap["tb_grupo"][["CO_GRUPO", "NO_GRUPO"]], on="CO_GRUPO", how="left")
    .merge(
        sigtap["tb_sub_grupo"][["CO_GRUPO", "CO_SUB_GRUPO", "NO_SUB_GRUPO"]],
        on=["CO_GRUPO", "CO_SUB_GRUPO"], how="left",
    )
    .merge(
        sigtap["tb_forma_organizacao"][
            ["CO_GRUPO", "CO_SUB_GRUPO", "CO_FORMA_ORGANIZACAO", "NO_FORMA_ORGANIZACAO"]
        ],
        on=["CO_GRUPO", "CO_SUB_GRUPO", "CO_FORMA_ORGANIZACAO"], how="left",
    )
    .merge(
        sigtap["tb_financiamento"][["CO_FINANCIAMENTO", "NO_FINANCIAMENTO"]],
        on="CO_FINANCIAMENTO", how="left",
    )
)

# Domínio com chave repetida multiplicaria linhas e inflaria qualquer contagem.
assert len(procedimentos) == antes, f"merge multiplicou linhas: {antes} -> {len(procedimentos)}"

# TP_COMPLEXIDADE vem como código de um dígito.
COMPLEXIDADE = {"0": "Não se aplica", "1": "Atenção básica",
                "2": "Média complexidade", "3": "Alta complexidade"}
procedimentos["NO_COMPLEXIDADE"] = (
    procedimentos["TP_COMPLEXIDADE"].map(COMPLEXIDADE).fillna("(código novo)")
)

# Os três valores vêm em centavos, com zeros à esquerda.
for origem, destino in [("VL_SH", "VL_HOSPITALAR"),
                        ("VL_SP", "VL_PROFISSIONAL")]:
    procedimentos[destino] = pd.to_numeric(procedimentos[origem], errors="coerce") / 100
procedimentos["VL_TOTAL"] = (
    procedimentos["VL_HOSPITALAR"] + procedimentos["VL_PROFISSIONAL"]
)

print(f"Procedimentos: {len(procedimentos):,}")
for coluna in ["NO_GRUPO", "NO_SUB_GRUPO", "NO_FORMA_ORGANIZACAO", "NO_FINANCIAMENTO"]:
    print(f"  sem descrição em {coluna}: {int(procedimentos[coluna].isna().sum())}")

display(procedimentos[
    ["CO_PROCEDIMENTO", "NO_PROCEDIMENTO", "NO_GRUPO", "NO_COMPLEXIDADE", "VL_TOTAL"]
].head())

Procedimentos: 4,844
  sem descrição em NO_GRUPO: 0
  sem descrição em NO_SUB_GRUPO: 0
  sem descrição em NO_FORMA_ORGANIZACAO: 0
  sem descrição em NO_FINANCIAMENTO: 0


,CO_PROCEDIMENTO,NO_PROCEDIMENTO,NO_GRUPO,NO_COMPLEXIDADE,VL_TOTAL
0,0101010010,ATIVIDADE EDUCATIVA / ORIENTAÇÃO EM GRUPO NA A...,Ações de promoção e prevenção em saúde,Atenção básica,0.0
1,0101010028,ATIVIDADE EDUCATIVA / ORIENTAÇÃO EM GRUPO NA A...,Ações de promoção e prevenção em saúde,Média complexidade,0.0
2,0101010036,PRÁTICA CORPORAL / ATIVIDADE FÍSICA EM GRUPO,Ações de promoção e prevenção em saúde,Atenção básica,0.0
3,0101010095,PREVENÇÃO DA COVID-19 NAS ESCOLAS,Ações de promoção e prevenção em saúde,Atenção básica,0.0
4,0101010109,ATIVIDADES EDUCATIVAS DA POPULAÇÃO SOBRE A TEM...,Ações de promoção e prevenção em saúde,Atenção básica,0.0


## 8. Distribuição dos procedimentos

In [7]:
print("Procedimentos por grupo:")
display(
    procedimentos.groupby(["CO_GRUPO", "NO_GRUPO"])
    .agg(procedimentos=("CO_PROCEDIMENTO", "count"), valor_medio=("VL_TOTAL", "mean"))
    .round(2)
    .sort_values("procedimentos", ascending=False)
)

print("\nProcedimentos por complexidade:")
display(
    procedimentos.groupby("NO_COMPLEXIDADE")
    .agg(procedimentos=("CO_PROCEDIMENTO", "count"), valor_medio=("VL_TOTAL", "mean"))
    .round(2)
    .sort_values("procedimentos", ascending=False)
)

print("\nOs 10 procedimentos de maior valor:")
display(
    procedimentos.nlargest(10, "VL_TOTAL")[
        ["CO_PROCEDIMENTO", "NO_PROCEDIMENTO", "NO_COMPLEXIDADE", "VL_TOTAL"]
    ]
)

Procedimentos por grupo:


,,procedimentos,valor_medio
CO_GRUPO,NO_GRUPO,,
04,Procedimentos cirúrgicos,1690,1827.28
02,Procedimentos com finalidade diagnóstica,1075,36.69
03,Procedimentos clínicos,831,94.20
07,"Órteses, próteses e materiais especiais",531,1121.54
06,Medicamentos,389,60.49
05,"Transplantes de orgãos, tecidos e células",142,5994.26
01,Ações de promoção e prevenção em saúde,113,0.00
08,Ações complementares da atenção à saúde,45,161.06
09,Procedimentos para Ofertas de Cuidados Integrados,28,0.00



Procedimentos por complexidade:


,procedimentos,valor_medio
NO_COMPLEXIDADE,,
Média complexidade,2404,223.83
Alta complexidade,1604,2207.11
Não se aplica,640,945.38
Atenção básica,196,0.00



Os 10 procedimentos de maior valor:


,CO_PROCEDIMENTO,NO_PROCEDIMENTO,NO_COMPLEXIDADE,VL_TOTAL
3802,0505010020,TRANSPLANTE ALOGÊNICO DE CÉLULAS-TRONCO HEMATO...,Alta complexidade,71602.25
3804,0505010046,TRANSPLANTE ALOGÊNICO DE CÉLULAS-TRONCO HEMATO...,Alta complexidade,71602.25
3806,0505010062,TRANSPLANTE ALOGÊNICO DE CÉLULAS-TRONCO HEMATO...,Alta complexidade,71602.25
3815,0505020050,TRANSPLANTE DE FIGADO (ORGAO DE DOADOR FALECIDO),Alta complexidade,68838.89
3816,0505020068,TRANSPLANTE DE FIGADO (ORGAO DE DOADOR VIVO),Alta complexidade,68803.27
3821,0505020122,TRANSPLANTE DE PULMÃO BILATERAL,Alta complexidade,64434.67
3803,0505010038,TRANSPLANTE ALOGÊNICO DE CÉLULAS-TRONCO HEMATO...,Alta complexidade,58372.97
2644,0406030162,"IMPLANTE PERCUTÂNEO DE VÁLVULA AÓRTICA (TAVI),...",Alta complexidade,57000.00
3801,0505010011,TRANSPLANTE ALOGÊNICO DE CÉLULAS-TRONCO HEMATO...,Alta complexidade,54939.27
3805,0505010054,TRANSPLANTE ALOGÊNICO DE CÉLULAS-TRONCO HEMATO...,Alta complexidade,54939.27


## 9. Validação de qualidade

In [8]:
validacoes = pd.DataFrame([
    {"verificacao": "Procedimentos na competência", "valor": len(procedimentos)},
    {"verificacao": "Códigos duplicados",
     "valor": int(procedimentos["CO_PROCEDIMENTO"].duplicated().sum())},
    {"verificacao": "Código fora do padrão de 10 dígitos",
     "valor": int((procedimentos["CO_PROCEDIMENTO"].str.len() != 10).sum())},
    {"verificacao": "Sem nome de grupo", "valor": int(procedimentos["NO_GRUPO"].isna().sum())},
    {"verificacao": "Sem nome de subgrupo", "valor": int(procedimentos["NO_SUB_GRUPO"].isna().sum())},
    {"verificacao": "Sem forma de organização",
     "valor": int(procedimentos["NO_FORMA_ORGANIZACAO"].isna().sum())},
    {"verificacao": "Grupos distintos", "valor": int(procedimentos["CO_GRUPO"].nunique())},
    {"verificacao": "Procedimentos de alta complexidade",
     "valor": int((procedimentos["TP_COMPLEXIDADE"] == "3").sum())},
])

display(validacoes)

# O código tem de ser texto: com zero à esquerda, virar inteiro quebra o join
# com PROC_REA do SIH e PA_PROC_ID do SIA. O pandas 3.0 usa o dtype "str", e
# não mais "object", então a checagem tem de ser pelo tipo lógico.
assert pd.api.types.is_string_dtype(procedimentos["CO_PROCEDIMENTO"]), (
    "CO_PROCEDIMENTO precisa continuar texto"
)
com_zero = procedimentos["CO_PROCEDIMENTO"].str.startswith("0").sum()
print(f"\nProcedimentos com zero à esquerda: {com_zero:,} — é por isso que o código fica como texto.")

,verificacao,valor
0,Procedimentos na competência,4844
1,Códigos duplicados,0
2,Código fora do padrão de 10 dígitos,0
3,Sem nome de grupo,0
4,Sem nome de subgrupo,0
5,Sem forma de organização,0
6,Grupos distintos,9
7,Procedimentos de alta complexidade,1604



Procedimentos com zero à esquerda: 4,844 — é por isso que o código fica como texto.


## 10. Salvamento em Parquet

Saída em `dados/tratado/sigtap/`. Duas tabelas: a dimensão enriquecida, que é o que o resto do pipeline consome, e as tabelas de apoio da hierarquia, para quem quiser navegar por nível.

In [10]:
arquivos_gerados = []


def salvar(df: pd.DataFrame, nome: str) -> None:
    destino = DIRETORIO_TRATADO / nome
    df.to_parquet(destino, index=False)
    arquivos_gerados.append(destino)


COLUNAS_SAIDA = [
    "CO_PROCEDIMENTO", "NO_PROCEDIMENTO",
    "CO_GRUPO", "NO_GRUPO", "CO_SUB_GRUPO", "NO_SUB_GRUPO",
    "CO_FORMA_ORGANIZACAO", "NO_FORMA_ORGANIZACAO",
    "TP_COMPLEXIDADE", "NO_COMPLEXIDADE",
    "CO_FINANCIAMENTO", "NO_FINANCIAMENTO",
    "VL_HOSPITALAR", "VL_PROFISSIONAL", "VL_TOTAL",
    "QT_DIAS_PERMANENCIA", "QT_PONTOS", "DT_COMPETENCIA",
]

salvar(procedimentos[COLUNAS_SAIDA], f"sigtap_procedimentos_{COMPETENCIA}.parquet")
salvar(sigtap["tb_grupo"], f"sigtap_grupos_{COMPETENCIA}.parquet")
salvar(sigtap["tb_sub_grupo"], f"sigtap_subgrupos_{COMPETENCIA}.parquet")
salvar(sigtap["tb_forma_organizacao"], f"sigtap_formas_organizacao_{COMPETENCIA}.parquet")

total_kb = sum(a.stat().st_size for a in arquivos_gerados) / 1024
print(f"{len(arquivos_gerados)} arquivos, {total_kb:.0f} KB")
print(f"Pasta: {DIRETORIO_TRATADO}\n")
for arq in sorted(arquivos_gerados):
    print(f"  {arq.name:<44} {arq.stat().st_size / 1024:>8.1f} KB")

4 arquivos, 226 KB
Pasta: C:\Users\Vitor Nobre\Documents\Workspace\conecta-saude\dados\tratado\sigtap

  sigtap_formas_organizacao_202412.parquet         13.4 KB
  sigtap_grupos_202412.parquet                      2.6 KB
  sigtap_procedimentos_202412.parquet             205.0 KB
  sigtap_subgrupos_202412.parquet                   4.8 KB


## 11. Perguntas que o SIGTAP responde

Sozinho, o SIGTAP responde perguntas sobre a **regra**, não sobre a realidade:

1. Quantos procedimentos o SUS oferece, e como se distribuem por complexidade?
2. Quais procedimentos são de alta complexidade e portanto concentram-se em poucos municípios?
3. Quanto o SUS remunera cada procedimento?
4. Que fonte financia cada procedimento — atenção básica, MAC ou FAEC?
